IMPORT

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from preprocessing import URLProcessor
from feature_extraction import extract_batch
from models import (
    XGBoostBranch,
    build_bilstm_branch,
    build_hybrid_model,
)

from evaluation import (
    evaluate_model,
    measure_latency,
    plot_confusion_matrix,
    print_results,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

LOAD DATASET

In [3]:
df = pd.read_csv("malicious_phish.csv")

df.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


Label conversion

In [4]:
df["label"] = (
    df["type"] == "phishing"
).astype(int)

df["label"].value_counts()

label
0    557080
1     94111
Name: count, dtype: int64

TRAIN / TEST SPLIT

In [5]:
urls = df["url"].values
labels = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    urls,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 520952
Test: 130239


URL PREPROCESSING

In [6]:
processor = URLProcessor(
    max_len=150
)

X_train_seq = processor.fit_transform(
    X_train
)

X_test_seq = processor.transform(
    X_test
)

print(X_train_seq.shape)

(520952, 150)


FEATURE EXTRACTION

In [7]:
X_train_feat = extract_batch(
    X_train
)

X_test_feat = extract_batch(
    X_test
)

print(X_train_feat.shape)

(520952, 6)


TRAIN XGBOOST

In [8]:
xgb = XGBoostBranch()

start = time.time()

xgb.train(
    X_train_feat,
    y_train
)

xgb_train_time = time.time() - start

print("XGBoost train time:", xgb_train_time)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:11:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost train time: 3.573814630508423


XGBOOST PROBABILITY OUTPUT

In [9]:
xgb_prob_train = xgb.get_proba(
    X_train_feat
)

xgb_prob_test = xgb.get_proba(
    X_test_feat
)

print(xgb_prob_train.shape)

(520952, 1)


BUILD BiLSTM BRANCH

In [10]:
bilstm_input, bilstm_output = build_bilstm_branch(
    input_len=150,
    vocab_size=processor.get_vocab_size(),
    embed_dim=32,
    lstm_units=64,
    dropout_rate=0.3,
)

BUILD HYBRID MODEL

In [11]:
hybrid_model = build_hybrid_model(
    bilstm_input,
    bilstm_output,
    xgb_feature_dim=1,
    dropout_rate=0.3,
)

hybrid_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ url_input           │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 150, 32)   │     10,624 │ url_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     49,664 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xgb_input           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 65)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ xgb_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,112 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 70,689 (276.13 KB)

 Trainable params: 70,689 (276.13 KB)

 Non-trainable params: 0 (0.00 B)

TRAIN HYBRID MODEL

In [12]:
start = time.time()

history = hybrid_model.fit(
    [
        X_train_seq,
        xgb_prob_train
    ],
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

hybrid_train_time = time.time() - start

print("Hybrid train time:", hybrid_train_time)

Epoch 1/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 624s 85ms/step - accuracy: 0.9481 - loss: 0.1566 - val_accuracy: 0.9637 - val_loss: 0.1092
Epoch 2/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 856s 117ms/step - accuracy: 0.9669 - loss: 0.1049 - val_accuracy: 0.9707 - val_loss: 0.0891
Epoch 3/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 930s 126ms/step - accuracy: 0.9716 - loss: 0.0903 - val_accuracy: 0.9717 - val_loss: 0.0855
Epoch 4/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 912s 125ms/step - accuracy: 0.9749 - loss: 0.0804 - val_accuracy: 0.9756 - val_loss: 0.0757
Epoch 5/5
7326/7326 ━━━━━━━━━━━━━━━━━━━━ 870s 117ms/step - accuracy: 0.9769 - loss: 0.0738 - val_accuracy: 0.9772 - val_loss: 0.0697
Hybrid train time: 4192.584156036377
